# Paper Check Particle Life Plus: corrected analysis

Этот ноутбук проверяет **именно тот primary test, который обсуждался для paper check**.

Для каждого `optimized_run_idx = r` строится run-level величина

\[
\Delta_r = F_r^{\mathrm{optimized}} - \operatorname{median}\big(F_r^{\mathrm{random},1}, \ldots, F_r^{\mathrm{random},M}\big),
\]

где

\[
F_{anchor} = D(A,C) - D(A,B)
\]

где `A = control_a`, `B = control_b`, `C = walls`.
В stage-2 логах эта величина явно не записана, поэтому ноутбук досчитывает колонку `anchor_effect_minus_baseline`.

`A` — обычный rollout с seed `x`,  
`B` — обычный rollout с seed `x1`,  
`C` — rollout с тем же `x`, но с walls в первой половине.

Primary inference: **exact one-sided sign test** по независимым run-level `\Delta_r`.  
Все secondary metrics и per-distance tests ниже тоже считаются в той же anchor-style схеме.

Ноутбук дополнительно досчитывает **отдельные run-level тесты** для history-dependence distance metrics из старого offline analysis:
- `embedding_synced_cosine`
- `embedding_synced_euclidean`
- `embedding_cloud_chamfer_cosine`
- `delta_h_l2`
- `delta_h_mean_abs`
- `delta_h_dist_tau3000_wasserstein`
- `delta_h_dist_tau3000_ks`
- `delta_h_dist_tau3000_energy`
- `delta_h_dist_tau3000_wasserstein_zscore`
- `delta_h_dist_tau3000_ks_zscore`
- `delta_h_dist_tau3000_energy_zscore`


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import sys
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur] + list(cur.parents):
        if (candidate / '.git').exists() or (candidate / 'scripts').exists() or (candidate / 'experiments').exists():
            return candidate
    return start.resolve()


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis.history_dependence.paper_check_metric_stats import augment_rows_with_history_dependence_distances

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 200)

REPO_ROOT

In [ ]:
RESULT_ROOTS = [
    REPO_ROOT / 'experiments/paper_check_plife_plus/checkpoints',
]

OUTPUT_DIR = REPO_ROOT / 'analysis/results/paper_check_plife_plus_log_analysis_fixed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HD_ANALYSIS_CONFIG = REPO_ROOT / 'experiments/paper_check_plife_plus/frustration_simulation/analysis_config.yaml'

PRIMARY_METRIC = 'anchor_effect_minus_baseline'
SECONDARY_METRICS = [
    'baseline_distance',
    'walls_effect_distance_ctrl_a',
    'anchor_effect_over_baseline_ratio',
    'clip_oe_loss_walls_minus_control_a',
    'msc_score_walls_minus_control_a',
    'msc_loss_walls_minus_control_a',
    'msc_score_anchor_absdiff_minus_baseline',
    'msc_loss_anchor_absdiff_minus_baseline',
]

CONTROL_KINDS = ('random',)
CONTROL_AGG = 'median'

STRICT_EXPERIMENT_SHAPE = True
EXPECTED_OPTIMIZED_PER_RUN = 1
EXPECTED_INIT_PER_RUN = 0
EXPECTED_RANDOM_PER_RUN = None  # infer from data / do not hard-restrict

RESULT_ROOTS

In [ ]:
def resolve_frustration_root(root: Path) -> Path:
    root = Path(root)
    if (root / 'trial_results.csv').exists() or (root / 'trial_data').exists():
        return root
    if (root / 'frustration_simulation').exists():
        return root / 'frustration_simulation'
    return root


def load_trial_rows(root: Path) -> pd.DataFrame:
    fs_root = resolve_frustration_root(root)
    csv_path = fs_root / 'trial_results.csv'
    summary_path = fs_root / 'summary.json'
    rows = []
    source_summary = None

    if summary_path.exists():
        source_summary = json.loads(summary_path.read_text())

    if csv_path.exists():
        df = pd.read_csv(csv_path)
    else:
        trial_data_dir = fs_root / 'trial_data'
        for path in sorted(trial_data_dir.glob('trial_*.json')):
            rows.append(json.loads(path.read_text()))
        df = pd.DataFrame(rows)

    if df.empty:
        return df

    df = df.copy()
    df['source_root'] = str(root)
    df['source_name'] = Path(root).name
    df['frustration_root'] = str(fs_root)

    if source_summary is not None:
        for key, value in source_summary.items():
            col = f'summary__{key}'
            if col not in df.columns:
                df[col] = value
    return df


def coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            try:
                out[col] = pd.to_numeric(out[col])
            except Exception:
                pass
    return out


def canonicalize_candidate_kind(kind, label=None):
    tokens = []
    if kind is not None and not (isinstance(kind, float) and np.isnan(kind)):
        tokens.append(str(kind).strip().lower())
    if label is not None and not (isinstance(label, float) and np.isnan(label)):
        tokens.append(str(label).strip().lower())

    joined = ' '.join(tokens)

    if any(t in joined for t in ['optimized', 'best', 'opt']):
        return 'optimized'
    if any(t in joined for t in ['init', 'initial', 'start']):
        return 'init'
    if any(t in joined for t in ['random', 'rand']):
        return 'random'
    return 'other'


def derive_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'effect_minus_baseline' not in out.columns:
        if {'walls_effect_distance', 'baseline_distance'}.issubset(out.columns):
            out['effect_minus_baseline'] = out['walls_effect_distance'] - out['baseline_distance']
    if 'anchor_effect_minus_baseline' not in out.columns:
        if {'walls_effect_distance_ctrl_a', 'baseline_distance'}.issubset(out.columns):
            out['anchor_effect_minus_baseline'] = out['walls_effect_distance_ctrl_a'] - out['baseline_distance']
    if 'effect_over_baseline_ratio' not in out.columns:
        if {'walls_effect_distance', 'baseline_distance'}.issubset(out.columns):
            denom = pd.to_numeric(out['baseline_distance'], errors='coerce').astype(float)
            numer = pd.to_numeric(out['walls_effect_distance'], errors='coerce').astype(float)
            out['effect_over_baseline_ratio'] = numer / (denom + 1e-12)
    if 'anchor_effect_over_baseline_ratio' not in out.columns:
        if {'walls_effect_distance_ctrl_a', 'baseline_distance'}.issubset(out.columns):
            denom = pd.to_numeric(out['baseline_distance'], errors='coerce').astype(float)
            numer = pd.to_numeric(out['walls_effect_distance_ctrl_a'], errors='coerce').astype(float)
            out['anchor_effect_over_baseline_ratio'] = numer / (denom + 1e-12)
    if 'clip_oe_loss_walls_minus_control_a' not in out.columns:
        if {'clip_oe_loss_walls', 'clip_oe_loss_control_a'}.issubset(out.columns):
            out['clip_oe_loss_walls_minus_control_a'] = (
                pd.to_numeric(out['clip_oe_loss_walls'], errors='coerce')
                - pd.to_numeric(out['clip_oe_loss_control_a'], errors='coerce')
            )
    if 'msc_score_walls_minus_control_a' not in out.columns:
        if {'msc_score_walls', 'msc_score_control_a'}.issubset(out.columns):
            out['msc_score_walls_minus_control_a'] = (
                pd.to_numeric(out['msc_score_walls'], errors='coerce')
                - pd.to_numeric(out['msc_score_control_a'], errors='coerce')
            )
    if 'msc_loss_walls_minus_control_a' not in out.columns:
        if {'msc_loss_walls', 'msc_loss_control_a'}.issubset(out.columns):
            out['msc_loss_walls_minus_control_a'] = (
                pd.to_numeric(out['msc_loss_walls'], errors='coerce')
                - pd.to_numeric(out['msc_loss_control_a'], errors='coerce')
            )
    if 'msc_score_anchor_absdiff_minus_baseline' not in out.columns:
        if {'msc_score_walls', 'msc_score_control_a', 'msc_score_control_b'}.issubset(out.columns):
            score_w = pd.to_numeric(out['msc_score_walls'], errors='coerce')
            score_a = pd.to_numeric(out['msc_score_control_a'], errors='coerce')
            score_b = pd.to_numeric(out['msc_score_control_b'], errors='coerce')
            out['msc_score_anchor_absdiff_minus_baseline'] = (score_w - score_a).abs() - (score_b - score_a).abs()
    if 'msc_loss_anchor_absdiff_minus_baseline' not in out.columns:
        if {'msc_loss_walls', 'msc_loss_control_a', 'msc_loss_control_b'}.issubset(out.columns):
            loss_w = pd.to_numeric(out['msc_loss_walls'], errors='coerce')
            loss_a = pd.to_numeric(out['msc_loss_control_a'], errors='coerce')
            loss_b = pd.to_numeric(out['msc_loss_control_b'], errors='coerce')
            out['msc_loss_anchor_absdiff_minus_baseline'] = (loss_w - loss_a).abs() - (loss_b - loss_a).abs()
    return out


def add_anchor_style_metrics(df: pd.DataFrame, distance_base_names: list[str]) -> pd.DataFrame:
    out = df.copy()
    for base_name in distance_base_names:
        baseline_col = f'{base_name}__baseline_distance'
        anchor_col = f'{base_name}__walls_effect_distance_ctrl_a'
        effect_col = f'{base_name}__anchor_effect_minus_baseline'
        ratio_col = f'{base_name}__anchor_effect_over_baseline_ratio'
        if effect_col not in out.columns and {baseline_col, anchor_col}.issubset(out.columns):
            out[effect_col] = pd.to_numeric(out[anchor_col], errors='coerce') - pd.to_numeric(out[baseline_col], errors='coerce')
        if ratio_col not in out.columns and {baseline_col, anchor_col}.issubset(out.columns):
            denom = pd.to_numeric(out[baseline_col], errors='coerce').astype(float)
            numer = pd.to_numeric(out[anchor_col], errors='coerce').astype(float)
            out[ratio_col] = numer / (denom + 1e-12)
    return out


def add_namespace_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'frustration_root' in out.columns:
        experiment_key = out['frustration_root'].astype(str)
    elif 'source_root' in out.columns:
        experiment_key = out['source_root'].astype(str)
    else:
        experiment_key = pd.Series(['default'] * len(out), index=out.index, dtype=object)

    if 'source_name' in out.columns:
        experiment_label = out['source_name'].astype(str)
    else:
        experiment_label = experiment_key.astype(str)

    out['experiment_key'] = experiment_key
    out['experiment_label'] = experiment_label

    if 'optimized_run_idx' in out.columns:
        run_idx_numeric = pd.to_numeric(out['optimized_run_idx'], errors='coerce')
        run_idx_str = [
            f"run_{int(val):03d}" if pd.notna(val) else 'run_nan'
            for val in run_idx_numeric
        ]
        run_idx_series = pd.Series(run_idx_str, index=out.index, dtype=object)
        out['run_key'] = experiment_key.astype(str) + '::' + run_idx_series
        out['run_label'] = experiment_label.astype(str) + ' / ' + run_idx_series
    else:
        out['run_key'] = experiment_key.astype(str)
        out['run_label'] = experiment_label.astype(str)
    return out


def deduplicate_rows(rows: pd.DataFrame) -> pd.DataFrame:
    out = rows.copy()
    namespace_cols = [c for c in ['experiment_key', 'frustration_root', 'source_root'] if c in out.columns]
    if 'trial_idx' in out.columns:
        sort_cols = namespace_cols + ['trial_idx'] + [c for c in ['source_name'] if c in out.columns]
        subset_cols = namespace_cols + ['trial_idx']
        out = out.sort_values(sort_cols).drop_duplicates(subset=subset_cols, keep='last')
        return out.reset_index(drop=True)

    key_cols = namespace_cols + [c for c in ['optimized_run_idx', 'candidate_kind_raw', 'candidate_idx', 'candidate_label'] if c in out.columns]
    if key_cols:
        out = out.sort_values(key_cols + [c for c in ['source_name'] if c in out.columns]).drop_duplicates(subset=key_cols, keep='last')
    return out.reset_index(drop=True)


def bootstrap_mean_ci(x, *, n_boot=20000, seed=0, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, x.size, size=(n_boot, x.size))
    boot = x[idx].mean(axis=1)
    return tuple(np.quantile(boot, [alpha / 2, 1 - alpha / 2]))


def exact_sign_test_greater(deltas):
    d = pd.Series(deltas).dropna().astype(float).to_numpy()
    d_nz = d[np.abs(d) > 1e-12]
    n = int(d_nz.size)
    n_pos = int(np.sum(d_nz > 0))
    p = float(stats.binomtest(n_pos, n, 0.5, alternative='greater').pvalue) if n > 0 else np.nan
    return {'n_nonzero': n, 'n_positive': n_pos, 'p_value': p}


def wilcoxon_greater(deltas):
    d = pd.Series(deltas).dropna().astype(float).to_numpy()
    d_nz = d[np.abs(d) > 1e-12]
    if d_nz.size == 0:
        return np.nan
    try:
        return float(stats.wilcoxon(d_nz, alternative='greater').pvalue)
    except Exception:
        return np.nan


def summarize_deltas(deltas):
    d = pd.Series(deltas).dropna().astype(float).to_numpy()
    ci_low, ci_high = bootstrap_mean_ci(d, seed=0) if d.size else (np.nan, np.nan)
    sign = exact_sign_test_greater(d)
    return {
        'n_runs': int(d.size),
        'mean_delta': float(np.mean(d)) if d.size else np.nan,
        'median_delta': float(np.median(d)) if d.size else np.nan,
        'std_delta': float(np.std(d, ddof=1)) if d.size > 1 else np.nan,
        'ci95_mean_delta_low': float(ci_low),
        'ci95_mean_delta_high': float(ci_high),
        'n_positive': sign['n_positive'],
        'n_nonzero': sign['n_nonzero'],
        'sign_test_greater_p': sign['p_value'],
        'wilcoxon_greater_p': wilcoxon_greater(d),
    }


def benjamini_hochberg(p_values):
    p = np.asarray(pd.Series(p_values), dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    finite = np.flatnonzero(np.isfinite(p))
    if finite.size == 0:
        return out
    ranked = finite[np.argsort(p[finite])]
    ordered = p[ranked]
    n = float(ordered.size)
    q = ordered * n / np.arange(1, ordered.size + 1, dtype=float)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out[ranked] = np.clip(q, 0.0, 1.0)
    return out


def build_run_level_table(rows: pd.DataFrame, metric_cols: list[str], *, control_kinds=('init', 'random'), control_agg='median'):
    agg_fn = np.median if control_agg == 'median' else np.mean

    run_rows = []
    control_long_rows = []
    group_cols = [c for c in ['experiment_key', 'optimized_run_idx'] if c in rows.columns]
    if not group_cols:
        group_cols = ['optimized_run_idx']

    for group_value, sub in rows.groupby(group_cols, sort=True):
        sub = sub.copy()
        if isinstance(group_value, tuple):
            group_map = dict(zip(group_cols, group_value))
        else:
            group_map = {group_cols[0]: group_value}
        experiment_key = group_map.get('experiment_key', sub['experiment_key'].iloc[0] if 'experiment_key' in sub.columns else 'default')
        experiment_label = sub['experiment_label'].iloc[0] if 'experiment_label' in sub.columns else str(experiment_key)
        run_idx = group_map.get('optimized_run_idx', sub['optimized_run_idx'].iloc[0] if 'optimized_run_idx' in sub.columns else np.nan)
        run_label = sub['run_label'].iloc[0] if 'run_label' in sub.columns else f"{experiment_label} / run_{int(run_idx):03d}"
        run_key = sub['run_key'].iloc[0] if 'run_key' in sub.columns else f"{experiment_key}::run_{int(run_idx):03d}"
        opt = sub[sub['candidate_kind_canon'] == 'optimized'].copy()
        init = sub[sub['candidate_kind_canon'] == 'init'].copy()
        rnd = sub[sub['candidate_kind_canon'] == 'random'].copy()
        ctrl = sub[sub['candidate_kind_canon'].isin(control_kinds)].copy()

        if STRICT_EXPERIMENT_SHAPE:
            problems = []
            if EXPECTED_OPTIMIZED_PER_RUN is not None and len(opt) != EXPECTED_OPTIMIZED_PER_RUN:
                problems.append(f'optimized={len(opt)} (expected {EXPECTED_OPTIMIZED_PER_RUN})')
            if EXPECTED_INIT_PER_RUN is not None and len(init) != EXPECTED_INIT_PER_RUN:
                problems.append(f'init={len(init)} (expected {EXPECTED_INIT_PER_RUN})')
            if EXPECTED_RANDOM_PER_RUN is not None and len(rnd) != EXPECTED_RANDOM_PER_RUN:
                problems.append(f'random={len(rnd)} (expected {EXPECTED_RANDOM_PER_RUN})')
            if problems:
                raise ValueError(f'Run {run_idx}: bad experiment shape: ' + ', '.join(problems))

        if opt.empty:
            warnings.warn(f'Run {run_idx}: no optimized row, skipping.')
            continue
        if ctrl.empty:
            warnings.warn(f'Run {run_idx}: no control rows, skipping.')
            continue

        if len(opt) > 1:
            sort_cols = [c for c in ['trial_idx', 'candidate_idx'] if c in opt.columns]
            if sort_cols:
                opt = opt.sort_values(sort_cols)
            opt = opt.tail(1)

        opt_row = opt.iloc[0]
        row = {
            'experiment_key': experiment_key,
            'experiment_label': experiment_label,
            'run_key': run_key,
            'run_label': run_label,
            'optimized_run_idx': run_idx,
            'n_control_rows': int(len(ctrl)),
            'n_init_rows': int(len(init)),
            'n_random_rows': int(len(rnd)),
        }

        for extra_col in ['source_name', 'source_root', 'frustration_root', 'seed_x', 'seed_x1', 'x_seed', 'x1_seed']:
            if extra_col in opt_row.index:
                row[f'optimized__{extra_col}'] = opt_row[extra_col]

        for metric in metric_cols:
            row[f'{metric}__optimized'] = opt_row.get(metric, np.nan)

            init_vals = pd.to_numeric(init.get(metric, pd.Series(dtype=float)), errors='coerce').dropna().to_numpy()
            rnd_vals = pd.to_numeric(rnd.get(metric, pd.Series(dtype=float)), errors='coerce').dropna().to_numpy()
            ctrl_vals = pd.to_numeric(ctrl.get(metric, pd.Series(dtype=float)), errors='coerce').dropna().to_numpy()

            row[f'{metric}__init_median'] = float(np.median(init_vals)) if init_vals.size else np.nan
            row[f'{metric}__random_median'] = float(np.median(rnd_vals)) if rnd_vals.size else np.nan
            row[f'{metric}__random_mean'] = float(np.mean(rnd_vals)) if rnd_vals.size else np.nan
            row[f'{metric}__control_{control_agg}'] = float(agg_fn(ctrl_vals)) if ctrl_vals.size else np.nan
            row[f'{metric}__delta_vs_control_{control_agg}'] = row[f'{metric}__optimized'] - row[f'{metric}__control_{control_agg}']

        for _, ctrl_row in ctrl.iterrows():
            ctrl_entry = {
                'experiment_key': experiment_key,
                'experiment_label': experiment_label,
                'run_key': run_key,
                'run_label': run_label,
                'optimized_run_idx': run_idx,
                'candidate_kind_canon': ctrl_row['candidate_kind_canon'],
                'candidate_kind_raw': ctrl_row.get('candidate_kind_raw', np.nan),
                'candidate_idx': ctrl_row.get('candidate_idx', np.nan),
                'candidate_label': ctrl_row.get('candidate_label', np.nan),
            }
            for metric in metric_cols:
                ctrl_entry[metric] = ctrl_row.get(metric, np.nan)
            control_long_rows.append(ctrl_entry)

        run_rows.append(row)

    run_table = pd.DataFrame(run_rows)
    if not run_table.empty:
        run_table = run_table.sort_values([c for c in ['experiment_label', 'optimized_run_idx'] if c in run_table.columns]).reset_index(drop=True)
    control_long = pd.DataFrame(control_long_rows)
    if not control_long.empty:
        control_long = control_long.sort_values([c for c in ['experiment_label', 'optimized_run_idx', 'candidate_kind_canon', 'candidate_idx'] if c in control_long.columns]).reset_index(drop=True)
    return run_table, control_long

In [ ]:
loaded_frames = []
missing_roots = []

for root in RESULT_ROOTS:
    root = Path(root)
    if not root.exists():
        missing_roots.append(root)
        continue
    frame = load_trial_rows(root)
    if frame.empty:
        warnings.warn(f'No trial rows found under {root}')
        continue
    loaded_frames.append(frame)

if missing_roots:
    print('Missing roots:')
    for path in missing_roots:
        print('  -', path)

if not loaded_frames:
    raise FileNotFoundError('No logs found in RESULT_ROOTS. Update RESULT_ROOTS in the config cell.')

rows = pd.concat(loaded_frames, ignore_index=True)
rows = coerce_numeric(rows)

rows['candidate_kind_raw'] = rows['candidate_kind'] if 'candidate_kind' in rows.columns else np.nan
rows['candidate_kind_canon'] = [
    canonicalize_candidate_kind(kind, label)
    for kind, label in zip(
        rows['candidate_kind_raw'] if 'candidate_kind_raw' in rows.columns else [None] * len(rows),
        rows['candidate_label'] if 'candidate_label' in rows.columns else [None] * len(rows),
    )
]

rows = add_namespace_columns(rows)
rows = derive_metrics(rows)
rows = deduplicate_rows(rows)
rows, _, hd_distance_base_names = augment_rows_with_history_dependence_distances(
    rows,
    analysis_config_path=HD_ANALYSIS_CONFIG,
)
rows = add_anchor_style_metrics(rows, hd_distance_base_names)
hd_anchor_metric_cols = [
    f'{name}__anchor_effect_minus_baseline'
    for name in hd_distance_base_names
    if f'{name}__anchor_effect_minus_baseline' in rows.columns
]

metric_cols = [
    c
    for c in dict.fromkeys([PRIMARY_METRIC] + SECONDARY_METRICS + hd_anchor_metric_cols)
    if c in rows.columns
]

display(Markdown('### Loaded rows'))
display(rows.head())

display(Markdown('### Available candidate kinds'))
display(rows['candidate_kind_canon'].value_counts(dropna=False).rename_axis('candidate_kind_canon').to_frame('count'))

display(Markdown('### Metric columns used'))
display(pd.DataFrame({'metric': metric_cols}))

display(Markdown('### History-dependence distance metrics added from raw artifacts'))
pd.DataFrame({'distance_name': hd_distance_base_names, 'effect_metric_column': [f'{name}__anchor_effect_minus_baseline' for name in hd_distance_base_names]})

## Sanity checks

In [ ]:
if 'optimized_run_idx' not in rows.columns:
    raise KeyError("Expected column 'optimized_run_idx' in trial rows.")

coverage = (
    rows.groupby(['experiment_label', 'optimized_run_idx', 'candidate_kind_canon'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .sort_values(['experiment_label', 'optimized_run_idx'])
    .reset_index(drop=True)
)

display(Markdown('### Coverage per run'))
display(coverage)

ignored = rows.loc[~rows['candidate_kind_canon'].isin(['optimized', 'init', 'random'])]
if not ignored.empty:
    display(Markdown('### Ignored rows with unsupported candidate kinds'))
    display(ignored[['experiment_label', 'optimized_run_idx', 'candidate_kind_raw', 'candidate_label']].drop_duplicates().reset_index(drop=True))

## Run-level table: optimized vs control median

In [ ]:
run_table, control_long = build_run_level_table(
    rows,
    metric_cols=metric_cols,
    control_kinds=CONTROL_KINDS,
    control_agg=CONTROL_AGG,
)

run_table.to_csv(OUTPUT_DIR / 'run_level_table.csv', index=False)
control_long.to_csv(OUTPUT_DIR / 'control_rows_long.csv', index=False)

primary_cols = [
    'experiment_label',
    'run_label',
    'optimized_run_idx',
    'n_init_rows',
    'n_random_rows',
    f'{PRIMARY_METRIC}__optimized',
    f'{PRIMARY_METRIC}__init_median',
    f'{PRIMARY_METRIC}__random_median',
    f'{PRIMARY_METRIC}__control_{CONTROL_AGG}',
    f'{PRIMARY_METRIC}__delta_vs_control_{CONTROL_AGG}',
]
primary_cols = [c for c in primary_cols if c in run_table.columns]

display(Markdown('### Primary run-level table'))
display(run_table[primary_cols])

## Primary hypothesis test

In [ ]:
primary_delta_col = f'{PRIMARY_METRIC}__delta_vs_control_{CONTROL_AGG}'
if primary_delta_col not in run_table.columns:
    raise KeyError(f'Missing primary delta column: {primary_delta_col}')

primary_summary = pd.DataFrame([summarize_deltas(run_table[primary_delta_col])])
primary_summary.insert(0, 'primary_metric', PRIMARY_METRIC)
primary_summary.to_csv(OUTPUT_DIR / 'primary_test_summary.csv', index=False)

display(Markdown(
    f'''
Primary test:
\[
\Delta_r = {PRIMARY_METRIC}^{{opt}}_r - \operatorname{{{CONTROL_AGG}}}({PRIMARY_METRIC}^{{controls}}_r)
\]

Controls = `random` rows inside the same `(experiment_key, optimized_run_idx)` group.
Primary p-value = exact one-sided sign test over the run-level deltas.
'''
))
primary_summary

## Secondary metrics and per-distance tests

In [ ]:
secondary_records = []
for metric in metric_cols:
    delta_col = f'{metric}__delta_vs_control_{CONTROL_AGG}'
    if delta_col not in run_table.columns:
        continue
    rec = summarize_deltas(run_table[delta_col])
    rec['metric'] = metric
    secondary_records.append(rec)

secondary_summary = pd.DataFrame(secondary_records)
secondary_summary = secondary_summary[['metric', 'n_runs', 'mean_delta', 'median_delta', 'std_delta',
                                       'ci95_mean_delta_low', 'ci95_mean_delta_high',
                                       'n_positive', 'n_nonzero', 'sign_test_greater_p', 'wilcoxon_greater_p']]
if not secondary_summary.empty:
    secondary_summary['sign_test_greater_q_bh'] = benjamini_hochberg(secondary_summary['sign_test_greater_p'])
    secondary_summary['wilcoxon_greater_q_bh'] = benjamini_hochberg(secondary_summary['wilcoxon_greater_p'])
secondary_summary.to_csv(OUTPUT_DIR / 'secondary_metric_deltas.csv', index=False)

hd_distance_summary = secondary_summary.loc[
    secondary_summary['metric'].isin(hd_anchor_metric_cols)
].copy()
if not hd_distance_summary.empty:
    hd_distance_summary.insert(0, 'distance_name', hd_distance_summary['metric'].str.replace('__anchor_effect_minus_baseline$', '', regex=True))
hd_distance_summary.to_csv(OUTPUT_DIR / 'history_distance_metric_tests.csv', index=False)

display(Markdown('### History-dependence distance metric tests'))
display(hd_distance_summary)

display(Markdown('### All secondary metrics'))
secondary_summary

## Figures

In [ ]:
primary_opt_col = f'{PRIMARY_METRIC}__optimized'
primary_ctrl_col = f'{PRIMARY_METRIC}__control_{CONTROL_AGG}'
primary_delta_col = f'{PRIMARY_METRIC}__delta_vs_control_{CONTROL_AGG}'

plot_df = run_table.reset_index(drop=True).copy()
plot_df['plot_x'] = np.arange(len(plot_df), dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Left: per-run controls and optimized score
ax = axes[0]
for _, run_row in plot_df.iterrows():
    run_key = run_row['run_key']
    x = float(run_row['plot_x'])
    ctrl = control_long.loc[control_long['run_key'] == run_key, [PRIMARY_METRIC, 'candidate_kind_canon']].copy()
    ctrl = ctrl.dropna(subset=[PRIMARY_METRIC])
    x_ctrl = np.repeat(x - 0.12, len(ctrl))
    ax.scatter(x_ctrl, ctrl[PRIMARY_METRIC].to_numpy(), marker='o', alpha=0.8)
    ctrl_med = run_row[primary_ctrl_col]
    opt_val = run_row[primary_opt_col]
    ax.scatter([x], [ctrl_med], marker='s', s=70)
    ax.scatter([x + 0.12], [opt_val], marker='^', s=80)
    ax.plot([x, x + 0.12], [ctrl_med, opt_val], linewidth=1)

ax.set_title(f'{PRIMARY_METRIC}: controls vs optimized')
ax.set_xlabel('run')
ax.set_ylabel(PRIMARY_METRIC)
ax.set_xticks(plot_df['plot_x'])
ax.set_xticklabels(plot_df['run_label'], rotation=45, ha='right')

# Right: run-level deltas
ax = axes[1]
ax.axhline(0.0, linewidth=1)
ax.scatter(plot_df['plot_x'], plot_df[primary_delta_col], s=70)
for _, row in plot_df.iterrows():
    ax.text(row['plot_x'], row[primary_delta_col], f"{row[primary_delta_col]:.3g}", fontsize=9, ha='left', va='bottom')
ax.set_title(f'Run-level delta: optimized - control_{CONTROL_AGG}')
ax.set_xlabel('run')
ax.set_ylabel(primary_delta_col)
ax.set_xticks(plot_df['plot_x'])
ax.set_xticklabels(plot_df['run_label'], rotation=45, ha='right')

fig.tight_layout()
plt.show()

## Saved artifacts

Ноутбук сохраняет в `analysis/results/paper_check_plife_plus_log_analysis_fixed`:

- `run_level_table.csv`
- `control_rows_long.csv`
- `primary_test_summary.csv`
- `secondary_metric_deltas.csv`
- `history_distance_metric_tests.csv`

Главное, что надо смотреть первым:
1. `run_level_table.csv`
2. `primary_test_summary.csv`
3. `history_distance_metric_tests.csv`
4. plot с `anchor_effect_minus_baseline__delta_vs_control_median`
